# 15. curate and the descriptors: exercises

Exercises of [chapter 15](../../15_curate.md). Run the setup cell first.

Work in a copy of this notebook, under [`docs/manual/solutions/`](..), in a folder `solutions_<you>`: it is regenerated by [`docs/manual/tools/build_notebooks.py`](../../tools/build_notebooks.py), and its links are written for that folder.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import copy, datetime, shutil, tempfile
import numpy as np
import pystac, rasterio, yaml
from rasterio.transform import from_origin
from data_acquisition.catalog import build, descriptors, stac
from data_acquisition.catalog import summary as summaries
from data_acquisition.readers import geotiff, vector
from data_acquisition.steps import curate

REFERENCE = yaml.safe_load((ROOT / "docs/manual/solutions/training_pond.yaml").read_text())


def load(s):
    path = Path(tempfile.mkdtemp()) / "summary.yaml"
    path.write_text(yaml.safe_dump(copy.deepcopy(s), sort_keys=False))
    return summaries.load(str(path))


def scratch_root():
    root = Path(tempfile.mkdtemp()) / "root"
    shutil.copytree(SAMPLE, root, ignore=shutil.ignore_patterns("stac"))
    for p in root.rglob("*"):          # writable copies: the sample's files are read-only like downloads
        if p.is_file():
            p.chmod(0o644)
    return root


s = load(REFERENCE)
catalog, _ = build.build(SAMPLE, s)
items = {i.id.split("_")[-1]: i for i in catalog.get_items(recursive=True)}
print(sorted(items))

**1. Descriptors.** On a copy of one sample item, set `colonies` as `manual` without `by`, as `claude` with `by`, and an unknown field. Which calls fail? What does `descriptors.missing` return before and after?

> **From the chapter.** §1: every descriptor records its source, and a value of None is missing.
>
> **Code.** `descriptors.set_field` and `descriptors.missing`, in [`data_acquisition/catalog/descriptors.py`](../../../../data_acquisition/catalog/descriptors.py). `items` is defined above: the sample's built items, by product.

In [ ]:
item = items["orthomosaic"].clone()
...  # your code here (descriptors.set_field, descriptors.missing)

**2. Seasons.** Compute the season of 15 January, 15 April, 15 July and 15 October at 48° N and at 45° S. Then run `curate._season` on the sample's items: why is the DTM's season `summer`?

> **From the chapter.** §3, on seasons: whole months, flipped south of the equator.
>
> **Code.** `descriptors.season` in [`data_acquisition/catalog/descriptors.py`](../../../../data_acquisition/catalog/descriptors.py), and `curate._season` in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py).

In [ ]:
...  # your code here (descriptors.season, curate._season)

**3. Make checks fire.** In a scratch copy of the sample, add a five-band multispectral raster with values 0-255 (digital numbers, not reflectance) and a DSM whose empty cells are -32767 without a declared nodata, describe them, build and curate. Which flags appear?

> **From the chapter.** §3: the table of checks, with their thresholds.
>
> **Code.** `scratch_root()` and `load(s)`, defined above; `build.build` in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py), and `curate.curate_item` in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py).

In [ ]:
root = scratch_root()
folder = root / "repo/doi_10.5555_training.pond"
...  # your code here (rasterio: write the two rasters; add two entries to a copy of the summary)

**4. Verification.** For each sample item, which source would `_checksum_verdict` use? Then reproduce finding 20 in a scratch copy: verify, change the DTM, verify again.

> **From the chapter.** §4: the order of verification, and what a mismatch leaves behind.
>
> **Code.** `curate._checksum_verdict` and `curate.curate`, in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py); `stac.asset_path`, `stac.save` and `stac.load` in [`data_acquisition/catalog/stac.py`](../../../../data_acquisition/catalog/stac.py).

In [ ]:
...  # your code here (curate._checksum_verdict(asset, path, provenance))

**5. Add a check.** Write, outside the pipeline, a check `no_overviews` for rasters wider or taller than a limit that have no overviews (slow to display; milestone 2 will need quick-looks), apply it after `curate_item`, and test it with a small limit.

> **From the chapter.** §5: what makes a check part of the pipeline. Here it stays outside.
>
> **Code.** `curate.curate_item` in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py), `descriptors.add_quality` in [`data_acquisition/catalog/descriptors.py`](../../../../data_acquisition/catalog/descriptors.py), `geotiff.info` in [`data_acquisition/readers/geotiff.py`](../../../../data_acquisition/readers/geotiff.py), and `stac.asset_path` in [`data_acquisition/catalog/stac.py`](../../../../data_acquisition/catalog/stac.py).

In [ ]:
def check_overviews(item, key, limit=4096):
    ...  # your code here (geotiff.info(...)["overviews"], descriptors.add_quality)

**6. One check at a time.** Call `curate.check_raster` on the DSM and `curate.check_vector` on the dams, first with the sample's site and then with a site in another UTM zone. Do the same for a vector item of the extract's dams. Then `curate.verify` on the orthomosaic and on the photos.

> **From the chapter.** §2: `curate_item` runs these checks in order, and §3 and §4 describe them.
>
> **Code.** `curate.check_raster`, `curate.check_vector` and `curate.verify`, in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py); `vector.info` in [`data_acquisition/readers/vector.py`](../../../../data_acquisition/readers/vector.py), and `stac.vector_item` in [`data_acquisition/catalog/stac.py`](../../../../data_acquisition/catalog/stac.py).

In [ ]:
site = s["sites"]["training_pond"]
other_zone = {**site, "crs": "EPSG:32611"}
fresh = {i.id.split("_")[-1]: i for i in build.build(SAMPLE, s)[0].get_items(recursive=True)}
...  # your code here

**7. Reading the result.** Curate every item, then read fields with `descriptors.get_field`, and print `descriptors.summary`. Where have you seen those rows before?

> **From the chapter.** §1: the descriptors and their sources.
>
> **Code.** `curate.curate_item` in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py), and `descriptors.get_field` and `descriptors.summary` in [`data_acquisition/catalog/descriptors.py`](../../../../data_acquisition/catalog/descriptors.py).

In [ ]:
...  # your code here